# **Function Calling** with Responses API (Azure OpenAI)

This notebook demonstrates how to use **Function Calling** with the Responses API on **Azure OpenAI** so the model can **select** and **invoke** tools (functions) you expose, while **you** execute those functions in Python and **return** the results to the model using `previous_response_id` + `tool_outputs`.

---

## Goals

- Initialize the `AzureOpenAI` client  
- Define `function` tools with a JSON schema  
- Let the model pick tools (`tool_choice="auto"`)  
- **Execute** tool calls in Python and send the results back  
- Implement a tiny **agent loop** that repeats until there are no more tool calls

---

## Prerequisites

- Python 3.10+  
- `pip install openai`  
- Azure OpenAI Resource with a **model deployment** (e.g., `gpt‑4o-mini`, `gpt‑4.1-mini`, etc.)  
- Environment variables:
  - `AZURE_OPENAI_API_KEY`
  - `AZURE_OPENAI_ENDPOINT` (e.g., `https://<your-resource>.openai.azure.com`)
  - `MODEL_DEPLOYMENT_NAME` (the **deployment name** of your model on Azure)

> ℹ️ **Endpoint note for Responses v1**  
> Use `base_url = <endpoint>/openai/v1/` with the OpenAI SDK for the latest Responses API surface on Azure.

## [Disclaimer by OpenAI](https://platform.openai.com/docs/assistants/migration?user-chat-app=responses)
*After achieving feature parity in the Responses API, we've deprecated the Assistants API. It will shut down on August 26, 2026. Follow the guidance below to update your integration. [Learn more](https://platform.openai.com/docs/guides/migrate-to-responses).*

# Constants and Libraries

In [1]:
import os, json
from pprint import pprint
from IPython.display import Markdown, Image, display
from dotenv import load_dotenv
from openai import AzureOpenAI

if not load_dotenv("./../../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

instructions = "You are a helpful assistant. Use tools when needed to fetch facts or perform calculations."

double_question = """
What's the weather in Milan today (in °C)?
Also, if a Margherita pizza costs 12, what's the price after a 15% discount?"""

Environment variables have been loaded ;-)


In [2]:
# Initialize client
client = AzureOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

print(f"OpenAI Endpoint base_url: {client.base_url}")

OpenAI Endpoint base_url: https://mmoaiswc-01.openai.azure.com/openai/


# Define tools (Function Calling)
We’ll expose two sample functions:
- `get_weather(location, unit)` → returns mock weather
- `calculate_discount(price, percent)` → computes a discounted price

## Tools **definition**

In [3]:
tools = [
    {
        "type": "function",
        "name": "get_weather",  # <-- top-level name
        "description": "Get current weather for a specific location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["c", "f"], "description": "Temperature unit (c/f)"}
            },
            "required": ["location"]
        }
    },
    {
        "type": "function",
        "name": "calculate_discount",  # <-- top-level name
        "description": "Calculate the discounted price.",
        "parameters": {
            "type": "object",
            "properties": {
                "price":   {"type": "number", "description": "Original price"},
                "percent": {"type": "number", "description": "Discount percentage (0-100)"}
            },
            "required": ["price", "percent"]
        }
    }
]

tools

[{'type': 'function',
  'name': 'get_weather',
  'description': 'Get current weather for a specific location.',
  'parameters': {'type': 'object',
   'properties': {'location': {'type': 'string', 'description': 'City name'},
    'unit': {'type': 'string',
     'enum': ['c', 'f'],
     'description': 'Temperature unit (c/f)'}},
   'required': ['location']}},
 {'type': 'function',
  'name': 'calculate_discount',
  'description': 'Calculate the discounted price.',
  'parameters': {'type': 'object',
   'properties': {'price': {'type': 'number', 'description': 'Original price'},
    'percent': {'type': 'number',
     'description': 'Discount percentage (0-100)'}},
   'required': ['price', 'percent']}}]

## Tools **implementation** (client-side)
Note: we use mock data to avoid external dependencies.
In production, these functions can call real APIs, databases, etc.

In [4]:
def get_weather(location: str, unit: str = "c"):
    # Mock data
    location_length = len(location)
    if location_length % 2 == 0: # even number
        base = {"location": location, "condition": "Sunny"}
    else: # odd number
        base = {"location": location, "condition": "Cloudy"}
    if unit.lower() == "f":
        base["temp"] = location_length*3  # ~12°C
        base["unit"] = "°F"
    else:
        base["temp"] = location_length
        base["unit"] = "°C"
    return base

def calculate_discount(price: float, percent: float):
    if percent < 0 or percent > 100:
        raise ValueError("percent must be in [0, 100]")
    final = round(price * (1 - percent/100.0), 2)
    return {"original": price, "percent": percent, "final": final, "currency": "EUR"}

print(get_weather("Cavi di Lavagna"))
print(calculate_discount(100,15))

{'location': 'Cavi di Lavagna', 'condition': 'Cloudy', 'temp': 15, 'unit': '°C'}
{'original': 100, 'percent': 15, 'final': 85.0, 'currency': 'EUR'}


# First turn: let the model decide if it should call tools
We’ll set `tool_choice="auto"` so the model can decide which tools to call, and even call more than one if parallel_tool_calls=True.

In [5]:
prompt = double_question

resp1 = client.responses.create(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    instructions=instructions,
    input=prompt,
    tools=tools,
    tool_choice="auto",           # let the model choose
    parallel_tool_calls=True      # optional: allow parallel tool calls
    # store=True                  # optional: server-side persistence
)

print(f"Response 1 ID: {resp1.id}")

Response 1 ID: resp_061e44fc8b9648eb00695a99e81fec8196b33f1d044a6a292a


## Check identified functions to run

In [6]:
print("=== RAW response (synthesis) ===")
# This prints a concise structure of the output array
functions_to_call = []
for item in getattr(resp1, "output", []) or []:
    functions_to_call.append({
        "call_id": getattr(item, "call_id", None),
        "function_name": getattr(item, "name", None),
        "arguments": getattr(item, "arguments", None),
        # "type": getattr(item, "type", None),
    })
pprint(functions_to_call)

=== RAW response (synthesis) ===
[{'arguments': '{"location":"Milan","unit":"c"}',
  'call_id': 'call_N2Vlh2V6jgASc0WI49yjxCcR',
  'function_name': 'get_weather'},
 {'arguments': '{"price":12,"percent":15}',
  'call_id': 'call_EjkQbYsnI0YZwgGqdRxnm5gL',
  'function_name': 'calculate_discount'}]


## Run the identified functions and collect their outputs

# Second run, passing the function outputs and collect the final result

In [7]:
def run_function(function_name, args_dict):
    # Build the Python expression string
    expr = f"{function_name}(**{args_dict})"
    # print("Evaluating:", expr)
    result = eval(expr)
    return result

run_function(function_name="get_weather", args_dict={"location": "Milano"})

tool_outputs = []

for f in functions_to_call:
    function_result = (run_function(function_name=f["function_name"], args_dict=f["arguments"]))
    tool_outputs.append({"type": "function_call_output", "call_id": f["call_id"], "output": json.dumps(function_result)})

pprint(tool_outputs)

[{'call_id': 'call_N2Vlh2V6jgASc0WI49yjxCcR',
  'output': '{"location": "Milan", "condition": "Cloudy", "temp": 5, "unit": '
            '"\\u00b0C"}',
  'type': 'function_call_output'},
 {'call_id': 'call_EjkQbYsnI0YZwgGqdRxnm5gL',
  'output': '{"original": 12, "percent": 15, "final": 10.2, "currency": "EUR"}',
  'type': 'function_call_output'}]


In [8]:
resp2 = client.responses.create(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    previous_response_id=resp1.id,  # link to the previous turn
    input = tool_outputs       # deliver the computed results
)

print(f"Response 2 ID: {resp2.id}")
print("\nFinal answer:")
print(resp2.output_text)

Response 2 ID: resp_061e44fc8b9648eb00695a99ea86e481969e5ac593b4a7a737

Final answer:
Today in Milan, it's cloudy with a temperature of 5°C. 

If the original price of a Margherita pizza is 12 EUR, after a 15% discount, the price would be 10.2 EUR.
